# 🧠 Brain Wave Explorer: Journey Into Your Mind

Welcome to the fascinating world of brain waves! Your brain is always creating electrical patterns, even when you sleep.

## What You'll Learn Today

In 45 minutes, you'll:
- Generate and explore different types of brain waves (EEG)
- Identify Delta, Theta, Alpha, and Beta waves
- Compute power spectrum to see frequency content
- Extract specific frequency bands
- Understand what brain waves tell us about mental states

## Skills You'll Master

- **Frequency analysis** - understanding signals in the frequency domain
- **Band extraction** - isolating specific frequency ranges
- **Power spectrum** - measuring signal strength at different frequencies
- **EEG interpretation** - connecting brain waves to mental states

**Ready to explore your brain's electrical language? Let's dive in!** 🌊

## Step 1: Prepare Your Lab Equipment

Let's load all the tools we'll need for brain wave exploration!

In [ ]:
# Import our brain analysis toolkit
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Make beautiful plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Brain wave analysis tools loaded!")
print("🧠 Ready to explore the electrical patterns of thought!")

---
# 🌊 Understanding Brain Waves

## What Are Brain Waves?

Your brain has about **86 billion neurons** (brain cells) that communicate using tiny electrical signals. When millions of neurons fire together, they create **brain waves** that we can measure with EEG (electroencephalogram) sensors.

## The Four Main Brain Wave Types

Scientists have identified different types of brain waves based on their **frequency** (how fast they oscillate):

| Wave Type | Frequency | Mental State | Symbol |
|-----------|-----------|--------------|--------|
| **Delta (δ)** | 0.5-4 Hz | Deep sleep, unconscious | 😴 |
| **Theta (θ)** | 4-8 Hz | Drowsy, light sleep, meditation | 💭 |
| **Alpha (α)** | 8-13 Hz | Relaxed, calm, eyes closed | 🧘 |
| **Beta (β)** | 13-30 Hz | Alert, focused, problem-solving | 🎯 |

There are also **Gamma waves** (30+ Hz) for intense focus, but we'll focus on the main four today.

## Hz = Hertz = Cycles Per Second

When we say "10 Hz," we mean the wave repeats 10 times every second. Delta waves are slow (0.5-4 cycles/second), while Beta waves are fast (13-30 cycles/second).

Let's create these waves and see what they look like!

In [ ]:
def generate_brain_wave(wave_type, duration=10, sampling_rate=256):
    """
    Generate realistic brain wave signals
    
    wave_type: 'delta', 'theta', 'alpha', 'beta', or 'mixed'
    duration: length in seconds
    sampling_rate: samples per second (256 Hz is standard for EEG)
    """
    time = np.linspace(0, duration, int(duration * sampling_rate))
    
    # Define frequency ranges and characteristics for each wave type
    wave_params = {
        'delta': {
            'freq_range': (0.5, 4),
            'name': 'Delta (Deep Sleep)',
            'emoji': '😴',
            'color': 'navy',
            'amplitude': 100  # Delta waves are large amplitude
        },
        'theta': {
            'freq_range': (4, 8),
            'name': 'Theta (Drowsy)',
            'emoji': '💭',
            'color': 'purple',
            'amplitude': 70
        },
        'alpha': {
            'freq_range': (8, 13),
            'name': 'Alpha (Relaxed)',
            'emoji': '🧘',
            'color': 'green',
            'amplitude': 50
        },
        'beta': {
            'freq_range': (13, 30),
            'name': 'Beta (Alert/Focused)',
            'emoji': '🎯',
            'color': 'orange',
            'amplitude': 30  # Beta waves are smaller amplitude
        }
    }
    
    params = wave_params[wave_type]
    freq_low, freq_high = params['freq_range']
    
    # Create signal with multiple frequency components in the range
    # (Real brain waves aren't pure sine waves - they're complex!)
    signal_data = np.zeros(len(time))
    
    # Add 3-5 frequency components within the band
    num_components = np.random.randint(3, 6)
    for _ in range(num_components):
        freq = np.random.uniform(freq_low, freq_high)
        phase = np.random.uniform(0, 2 * np.pi)
        amplitude = np.random.uniform(0.5, 1.5)
        signal_data += amplitude * np.sin(2 * np.pi * freq * time + phase)
    
    # Normalize and scale to realistic EEG amplitude (microvolts)
    signal_data = signal_data / np.max(np.abs(signal_data)) * params['amplitude']
    
    # Add amplitude modulation (brain waves naturally vary in strength)
    modulation = 1 + 0.3 * np.sin(2 * np.pi * 0.5 * time)
    signal_data = signal_data * modulation
    
    # Add realistic noise
    noise = np.random.normal(0, params['amplitude'] * 0.1, len(signal_data))
    signal_data = signal_data + noise
    
    return time, signal_data, params

print("✅ Brain wave generator ready!")

## Let's See All Four Brain Wave Types!

We'll create a 5-second sample of each wave type and display them together.

In [ ]:
# Generate all four wave types
wave_types = ['delta', 'theta', 'alpha', 'beta']
duration = 5  # seconds

fig, axes = plt.subplots(4, 1, figsize=(16, 10))

for ax, wave_type in zip(axes, wave_types):
    time, eeg_signal, params = generate_brain_wave(wave_type, duration=duration)
    
    ax.plot(time, eeg_signal, linewidth=1.5, color=params['color'])
    ax.set_ylabel('Amplitude (μV)', fontsize=10)
    ax.set_title(f"{params['emoji']} {params['name']} ({params['freq_range'][0]}-{params['freq_range'][1]} Hz)", 
                fontsize=13, fontweight='bold', color=params['color'])
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, duration)

axes[-1].set_xlabel('Time (seconds)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n🧠 Brain Wave Comparison:")
print("\n👀 Notice the differences:")
print("   • Delta waves are SLOW and BIG (deep sleep)")
print("   • Theta waves are slow (drowsy, daydreaming)")
print("   • Alpha waves are medium speed (relaxed, calm)")
print("   • Beta waves are FAST and small (alert, thinking)")
print("\n💡 Your brain produces different waves depending on what you're doing!")

## 💡 Try This!

**Experiment**: Change the `duration` variable above to 10 seconds. Can you see the wave patterns more clearly with a longer signal?

**Challenge**: Look closely at the plots. Which wave type oscillates (wiggles) the fastest? Which is slowest?

---
# 🔬 Frequency Analysis: The Power Spectrum

## What is a Power Spectrum?

When you look at a signal over time, it's hard to tell exactly what frequencies are present. The **power spectrum** shows us the strength of each frequency in the signal.

Think of it like this:
- **Time domain**: Shows how the signal changes moment to moment (what we've been plotting)
- **Frequency domain**: Shows which frequencies make up the signal (the power spectrum)

## Why Is This Useful?

Looking at the power spectrum lets us:
- Identify which brain wave types are present
- Measure how strong each frequency band is
- Detect changes in mental state
- Diagnose brain disorders

## The Tool: Fast Fourier Transform (FFT)

The FFT is a mathematical algorithm that converts signals from time domain to frequency domain. It's like a prism that splits white light into a rainbow of colors!

Let's compute the power spectrum for an Alpha wave!

In [ ]:
# Generate a 30-second alpha wave signal
time_alpha, signal_alpha, params_alpha = generate_brain_wave('alpha', duration=30)
sampling_rate = 256  # Hz

# Compute power spectrum using FFT
# FFT = Fast Fourier Transform
fft_values = fft(signal_alpha)
fft_freq = fftfreq(len(signal_alpha), 1/sampling_rate)

# Calculate power (magnitude squared)
power = np.abs(fft_values) ** 2

# Only keep positive frequencies (negative frequencies are mirror images)
positive_freq_mask = fft_freq > 0
frequencies = fft_freq[positive_freq_mask]
power_spectrum = power[positive_freq_mask]

# Plot time domain and frequency domain side by side
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Time domain (first 5 seconds)
time_mask = time_alpha <= 5
ax1.plot(time_alpha[time_mask], signal_alpha[time_mask], 
         linewidth=2, color='green', label='Alpha Wave')
ax1.set_xlabel('Time (seconds)', fontsize=12)
ax1.set_ylabel('Amplitude (μV)', fontsize=12)
ax1.set_title('🧘 Time Domain: Alpha Wave (8-13 Hz)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

# Frequency domain (0-50 Hz range)
freq_mask = frequencies < 50
ax2.plot(frequencies[freq_mask], power_spectrum[freq_mask], 
         linewidth=2, color='green', label='Power Spectrum')
ax2.fill_between(frequencies[freq_mask], power_spectrum[freq_mask], 
                 alpha=0.3, color='green')

# Highlight the alpha band (8-13 Hz)
ax2.axvspan(8, 13, alpha=0.2, color='yellow', label='Alpha Band (8-13 Hz)')
ax2.set_xlabel('Frequency (Hz)', fontsize=12)
ax2.set_ylabel('Power', fontsize=12)
ax2.set_title('📊 Frequency Domain: Power Spectrum', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

print("\n🔍 What do you notice?")
print("   • The power spectrum has a big peak in the 8-13 Hz range")
print("   • This confirms our signal is mainly Alpha waves!")
print("   • The yellow highlighted area shows the Alpha frequency band")

## Compare All Four Wave Types!

Let's see how the power spectrum looks different for each brain wave type.

In [ ]:
# Function to compute and plot power spectrum
def compute_power_spectrum(signal_data, sampling_rate=256, max_freq=50):
    """Compute power spectrum from signal"""
    fft_vals = fft(signal_data)
    freqs = fftfreq(len(signal_data), 1/sampling_rate)
    power = np.abs(fft_vals) ** 2
    
    # Keep only positive frequencies
    mask = (freqs > 0) & (freqs < max_freq)
    return freqs[mask], power[mask]

# Generate and analyze all four wave types
fig, axes = plt.subplots(4, 1, figsize=(16, 12))

wave_types = ['delta', 'theta', 'alpha', 'beta']
band_ranges = [(0.5, 4), (4, 8), (8, 13), (13, 30)]

for ax, wave_type, (low, high) in zip(axes, wave_types, band_ranges):
    # Generate signal
    time_wave, signal_wave, params = generate_brain_wave(wave_type, duration=30)
    
    # Compute power spectrum
    freqs, power = compute_power_spectrum(signal_wave)
    
    # Plot
    ax.plot(freqs, power, linewidth=2, color=params['color'])
    ax.fill_between(freqs, power, alpha=0.3, color=params['color'])
    
    # Highlight the relevant frequency band
    ax.axvspan(low, high, alpha=0.2, color='yellow', label=f'{params["name"]} band')
    
    ax.set_ylabel('Power', fontsize=11)
    ax.set_title(f"{params['emoji']} {params['name']} - Peak at {low}-{high} Hz", 
                fontsize=13, fontweight='bold', color=params['color'])
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 35)

axes[-1].set_xlabel('Frequency (Hz)', fontsize=12)
plt.tight_layout()
plt.show()

print("\n📊 Power Spectrum Analysis:")
print("\n🎯 Key Observations:")
print("   • Delta: Power concentrated at very low frequencies (0.5-4 Hz)")
print("   • Theta: Power in the 4-8 Hz range")
print("   • Alpha: Strong peak around 10 Hz (8-13 Hz range)")
print("   • Beta: Power spread across higher frequencies (13-30 Hz)")
print("\n💡 Each mental state has a unique frequency signature!")

---
# 🎯 Band Power: Measuring Brain Wave Strength

## What is Band Power?

**Band power** is the total power within a specific frequency range. It tells us how strong a particular brain wave type is.

For example:
- **Alpha band power** = total power in 8-13 Hz range
- **Beta band power** = total power in 13-30 Hz range

## Why Calculate Band Power?

Band power is useful for:
- Detecting mental states (relaxed vs. alert)
- Monitoring sleep stages
- Brain-computer interfaces
- Neurofeedback training
- Clinical diagnosis

Let's create a function to calculate band power for all four bands!

In [ ]:
def calculate_band_powers(signal_data, sampling_rate=256):
    """
    Calculate power in each frequency band
    
    Returns dictionary with power for delta, theta, alpha, and beta bands
    """
    # Define frequency bands
    bands = {
        'delta': (0.5, 4),
        'theta': (4, 8),
        'alpha': (8, 13),
        'beta': (13, 30)
    }
    
    # Compute power spectrum
    fft_vals = fft(signal_data)
    freqs = fftfreq(len(signal_data), 1/sampling_rate)
    power = np.abs(fft_vals) ** 2
    
    # Calculate power in each band
    band_powers = {}
    for band_name, (low_freq, high_freq) in bands.items():
        # Find frequencies in this band
        band_mask = (freqs >= low_freq) & (freqs <= high_freq)
        # Sum the power in this band
        band_powers[band_name] = np.sum(power[band_mask])
    
    return band_powers

# Test it on an alpha wave
time_test, signal_test, _ = generate_brain_wave('alpha', duration=30)
powers = calculate_band_powers(signal_test)

print("\n🧘 Analysis of Alpha Wave Signal:")
print("\n📊 Band Powers:")
for band, power in powers.items():
    print(f"   {band.capitalize():6s}: {power:>12,.0f}")

# Find dominant band
dominant_band = max(powers, key=powers.get)
print(f"\n🎯 Dominant band: {dominant_band.upper()}")
print(f"   This confirms our signal is primarily {dominant_band} waves!")

## Visualize Band Powers as a Bar Chart

Let's create a nice visualization showing the relative strength of each frequency band!

In [ ]:
def plot_band_powers(signal_data, title="Brain Wave Analysis", mental_state=""):
    """Create a bar chart of band powers"""
    powers = calculate_band_powers(signal_data)
    
    # Convert to percentages
    total_power = sum(powers.values())
    power_percentages = {band: (power/total_power)*100 for band, power in powers.items()}
    
    # Colors for each band
    colors = {'delta': 'navy', 'theta': 'purple', 'alpha': 'green', 'beta': 'orange'}
    emojis = {'delta': '😴', 'theta': '💭', 'alpha': '🧘', 'beta': '🎯'}
    
    # Create bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Absolute power
    bars1 = ax1.bar(powers.keys(), powers.values(), 
                    color=[colors[b] for b in powers.keys()],
                    alpha=0.7, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Absolute Power', fontsize=12)
    ax1.set_title(f'{title}\n{mental_state}', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add emoji labels
    ax1.set_xticks(range(len(powers)))
    ax1.set_xticklabels([f"{emojis[b]}\n{b.capitalize()}" for b in powers.keys()], fontsize=11)
    
    # Relative power (percentage)
    bars2 = ax2.bar(power_percentages.keys(), power_percentages.values(),
                    color=[colors[b] for b in power_percentages.keys()],
                    alpha=0.7, edgecolor='black', linewidth=2)
    ax2.set_ylabel('Relative Power (%)', fontsize=12)
    ax2.set_title('Relative Band Distribution', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.set_ylim(0, 100)
    
    # Add percentage labels on bars
    for bar, pct in zip(bars2, power_percentages.values()):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax2.set_xticks(range(len(power_percentages)))
    ax2.set_xticklabels([f"{emojis[b]}\n{b.capitalize()}" for b in power_percentages.keys()], fontsize=11)
    
    plt.tight_layout()
    plt.show()
    
    return powers, power_percentages

# Test on different wave types
print("Let's analyze different mental states!\n")

# Deep sleep (delta dominant)
time_sleep, signal_sleep, _ = generate_brain_wave('delta', duration=30)
powers_sleep, pct_sleep = plot_band_powers(signal_sleep, 
                                           title="😴 Deep Sleep State",
                                           mental_state="Delta waves dominate during deep sleep")

In [ ]:
# Relaxed meditation (alpha dominant)
time_relax, signal_relax, _ = generate_brain_wave('alpha', duration=30)
powers_relax, pct_relax = plot_band_powers(signal_relax,
                                           title="🧘 Relaxed Meditation State",
                                           mental_state="Alpha waves dominate when calm and relaxed")

In [ ]:
# Focused concentration (beta dominant)
time_focus, signal_focus, _ = generate_brain_wave('beta', duration=30)
powers_focus, pct_focus = plot_band_powers(signal_focus,
                                           title="🎯 Focused Concentration State",
                                           mental_state="Beta waves dominate during active thinking")

---
# 🌈 Real Brain Waves: Mixed States

## The Truth About Real Brain Waves

In reality, your brain doesn't produce just ONE type of wave at a time. Real EEG signals contain a **mixture** of different frequencies!

For example:
- **Awake and relaxed**: Mostly alpha, some beta
- **Light sleep**: Mix of theta and delta
- **Active thinking**: Mostly beta, some alpha

Let's create a more realistic mixed signal!

In [ ]:
def generate_mixed_eeg(duration=30, state='awake_relaxed'):
    """
    Generate realistic mixed-frequency EEG signals
    
    States:
    - 'awake_relaxed': Alpha dominant with some beta
    - 'awake_alert': Beta dominant with some alpha
    - 'drowsy': Theta dominant with some alpha
    - 'light_sleep': Theta and delta mix
    - 'deep_sleep': Delta dominant
    """
    # Define mixing ratios for each state
    state_configs = {
        'awake_relaxed': {
            'mix': {'alpha': 0.6, 'beta': 0.3, 'theta': 0.1},
            'name': 'Awake & Relaxed',
            'emoji': '🧘',
            'description': 'Eyes closed, calm, peaceful'
        },
        'awake_alert': {
            'mix': {'beta': 0.7, 'alpha': 0.2, 'theta': 0.1},
            'name': 'Awake & Alert',
            'emoji': '🎯',
            'description': 'Focused, problem-solving, active'
        },
        'drowsy': {
            'mix': {'theta': 0.6, 'alpha': 0.3, 'delta': 0.1},
            'name': 'Drowsy',
            'emoji': '💭',
            'description': 'Getting sleepy, daydreaming'
        },
        'light_sleep': {
            'mix': {'theta': 0.5, 'delta': 0.4, 'alpha': 0.1},
            'name': 'Light Sleep',
            'emoji': '😴',
            'description': 'Stage 1-2 sleep'
        },
        'deep_sleep': {
            'mix': {'delta': 0.8, 'theta': 0.2},
            'name': 'Deep Sleep',
            'emoji': '😴',
            'description': 'Stage 3-4 sleep, restorative'
        }
    }
    
    config = state_configs[state]
    sampling_rate = 256
    time = np.linspace(0, duration, int(duration * sampling_rate))
    
    # Generate mixed signal
    mixed_signal = np.zeros(len(time))
    
    for wave_type, weight in config['mix'].items():
        _, component, _ = generate_brain_wave(wave_type, duration=duration)
        mixed_signal += weight * component
    
    return time, mixed_signal, config

print("✅ Mixed EEG generator ready!")

In [ ]:
# Generate and analyze a realistic "awake and relaxed" state
time_mixed, signal_mixed, config_mixed = generate_mixed_eeg(duration=30, state='awake_relaxed')

# Show the signal
plt.figure(figsize=(16, 5))
plt.plot(time_mixed[:1280], signal_mixed[:1280], linewidth=1.5, color='darkblue')  # Show 5 seconds
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Amplitude (μV)', fontsize=12)
plt.title(f"{config_mixed['emoji']} {config_mixed['name']}: {config_mixed['description']}", 
         fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# Analyze band powers
print(f"\nAnalyzing: {config_mixed['name']}")
print(f"State: {config_mixed['description']}\n")
powers_mixed, pct_mixed = plot_band_powers(signal_mixed, 
                                           title=f"{config_mixed['emoji']} {config_mixed['name']}",
                                           mental_state=config_mixed['description'])

print("\n🎓 This is more like a REAL brain wave signal!")
print("   Real EEG always contains multiple frequency bands.")

---
# 🧪 Your Turn - Brain Wave Challenges!

Time to test your new brain wave analysis skills!

## Exercise 1: Identify the Mystery Signal 🕵️‍♀️

We'll generate a mystery brain wave signal. Your task is to figure out what mental state it represents by analyzing its band powers!

**Your task**:
1. Run the cell to generate a mystery signal
2. Calculate band powers
3. Identify which band is dominant
4. Guess the mental state!

In [ ]:
# Mystery signal!
import random
mystery_states = ['awake_relaxed', 'awake_alert', 'drowsy', 'light_sleep', 'deep_sleep']
mystery_state = random.choice(mystery_states)

# Generate (but don't reveal the state yet!)
time_mystery, signal_mystery, config_mystery = generate_mixed_eeg(duration=30, state=mystery_state)

# Show the signal
plt.figure(figsize=(16, 5))
plt.plot(time_mystery[:1280], signal_mystery[:1280], linewidth=1.5, color='purple')
plt.xlabel('Time (seconds)', fontsize=12)
plt.ylabel('Amplitude (μV)', fontsize=12)
plt.title('❓ Mystery EEG Signal - What mental state is this?', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

# YOUR TASK: Analyze this signal!
# 1. Calculate band powers
mystery_powers = calculate_band_powers(signal_mystery)

# 2. Plot the band powers
plot_band_powers(signal_mystery, title="❓ Mystery Signal Analysis")

# 3. Find the dominant band
dominant = max(mystery_powers, key=mystery_powers.get)
print(f"\n🔍 Your analysis shows that {dominant.upper()} waves are dominant.")
print(f"\n🤔 What mental state do you think this represents?")
print(f"   Hint: Look at which band has the highest percentage!\n")

# Reveal the answer
print(f"\n" + "="*60)
print(f"✅ ANSWER: {config_mystery['name']}")
print(f"   Description: {config_mystery['description']}")
print(f"="*60)

## Exercise 2: Track Sleep Stages 🌙

Sleep scientists use EEG to identify different sleep stages. Let's simulate a person falling asleep!

**Your task**: Generate signals for each sleep stage and observe how the brain waves change.

In [ ]:
# Simulate falling asleep: awake → drowsy → light sleep → deep sleep
sleep_stages = ['awake_relaxed', 'drowsy', 'light_sleep', 'deep_sleep']
stage_duration = 20  # seconds per stage

print("🌙 Sleep Progression Analysis\n")
print("Watch how brain waves change as you fall asleep...\n")

fig, axes = plt.subplots(4, 2, figsize=(16, 14))

for i, stage in enumerate(sleep_stages):
    # Generate signal for this stage
    time_stage, signal_stage, config_stage = generate_mixed_eeg(duration=stage_duration, state=stage)
    
    # Plot time-domain signal (left column)
    axes[i, 0].plot(time_stage[:1024], signal_stage[:1024], linewidth=1.5, color='darkblue')
    axes[i, 0].set_ylabel('Amplitude (μV)', fontsize=10)
    axes[i, 0].set_title(f"Stage {i+1}: {config_stage['emoji']} {config_stage['name']}", 
                        fontsize=12, fontweight='bold')
    axes[i, 0].grid(True, alpha=0.3)
    
    # Plot power spectrum (right column)
    freqs, power = compute_power_spectrum(signal_stage, max_freq=30)
    axes[i, 1].plot(freqs, power, linewidth=2, color='green')
    axes[i, 1].fill_between(freqs, power, alpha=0.3, color='green')
    axes[i, 1].set_ylabel('Power', fontsize=10)
    axes[i, 1].set_title(f"Power Spectrum", fontsize=12, fontweight='bold')
    axes[i, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Time (seconds)', fontsize=11)
axes[-1, 1].set_xlabel('Frequency (Hz)', fontsize=11)
plt.tight_layout()
plt.show()

print("\n🎓 Observations:")
print("   • As you fall asleep, the brain waves slow down")
print("   • Power spectrum shifts from higher to lower frequencies")
print("   • Deep sleep has the slowest, largest waves (delta)")
print("\n💡 Sleep scientists use this to monitor sleep quality!")

## Exercise 3: Build a Mental State Classifier 🧠

Create a simple function that automatically classifies mental state based on band powers!

**Your task**: Complete the classifier function below.

In [ ]:
def classify_mental_state(signal_data):
    """
    Classify mental state based on dominant frequency band
    
    Returns: (state, confidence, band_powers)
    """
    # Calculate band powers
    powers = calculate_band_powers(signal_data)
    
    # Convert to percentages
    total = sum(powers.values())
    percentages = {band: (power/total)*100 for band, power in powers.items()}
    
    # Find dominant band
    dominant_band = max(percentages, key=percentages.get)
    confidence = percentages[dominant_band]
    
    # Classify based on dominant band
    state_map = {
        'delta': ('Deep Sleep', '😴'),
        'theta': ('Drowsy/Light Sleep', '💭'),
        'alpha': ('Relaxed/Calm', '🧘'),
        'beta': ('Alert/Focused', '🎯')
    }
    
    state_name, emoji = state_map[dominant_band]
    
    return state_name, emoji, confidence, percentages

# Test the classifier on different signals
print("🧠 MENTAL STATE CLASSIFIER TEST\n")
print("="*70)

test_states = ['deep_sleep', 'drowsy', 'awake_relaxed', 'awake_alert']

for test_state in test_states:
    # Generate signal
    _, test_signal, true_config = generate_mixed_eeg(duration=30, state=test_state)
    
    # Classify
    pred_state, emoji, conf, pct = classify_mental_state(test_signal)
    
    print(f"\nTrue State: {true_config['name']}")
    print(f"Predicted: {emoji} {pred_state} (Confidence: {conf:.1f}%)")
    print(f"Band distribution: Delta={pct['delta']:.1f}% | Theta={pct['theta']:.1f}% | "
          f"Alpha={pct['alpha']:.1f}% | Beta={pct['beta']:.1f}%")
    print("-"*70)

print("\n✅ Classifier complete! It can identify mental states from EEG!")

---
# 🏆 Bonus Challenge: EEG Meditation Monitor

Create an interactive meditation quality monitor that rates how relaxed someone is based on their alpha wave power!

**Higher alpha = deeper relaxation**

In [ ]:
def meditation_quality_score(signal_data):
    """
    Rate meditation quality based on alpha wave dominance
    
    Returns score from 0-100
    """
    powers = calculate_band_powers(signal_data)
    total = sum(powers.values())
    
    # Alpha percentage is the main score
    alpha_pct = (powers['alpha'] / total) * 100
    
    # Bonus points for low beta (not thinking too much)
    beta_pct = (powers['beta'] / total) * 100
    
    # Score formula: alpha is good, beta is distracting
    score = alpha_pct * 1.5 - beta_pct * 0.5
    score = max(0, min(100, score))  # Clamp to 0-100
    
    return score, alpha_pct, beta_pct

# Simulate a meditation session (good, medium, poor)
print("🧘 MEDITATION QUALITY MONITOR\n")
print("="*70)

meditation_sessions = [
    ('Excellent meditation (deep relaxation)', 'awake_relaxed'),
    ('Moderate meditation (some thoughts)', 'drowsy'),
    ('Distracted (too much thinking)', 'awake_alert')
]

for session_name, state in meditation_sessions:
    # Generate 60-second session
    _, session_signal, _ = generate_mixed_eeg(duration=60, state=state)
    
    # Calculate score
    score, alpha, beta = meditation_quality_score(session_signal)
    
    # Rating
    if score >= 70:
        rating = "⭐⭐⭐⭐⭐ Excellent!"
    elif score >= 50:
        rating = "⭐⭐⭐⭐ Good"
    elif score >= 30:
        rating = "⭐⭐⭐ Fair"
    else:
        rating = "⭐⭐ Needs Practice"
    
    print(f"\n{session_name}")
    print(f"Score: {score:.1f}/100 | {rating}")
    print(f"Alpha power: {alpha:.1f}% | Beta power: {beta:.1f}%")
    
    # Progress bar
    bar_length = int(score / 2)
    bar = "█" * bar_length + "░" * (50 - bar_length)
    print(f"[{bar}]")
    print("-"*70)

print("\n💡 Use this to track meditation practice and improve relaxation skills!")

---
# 🎉 Congratulations, Brain Wave Explorer!

## What You Mastered Today

Incredible work! You now understand:

### 🌊 Brain Wave Types
- ✅ Delta waves (0.5-4 Hz) - Deep sleep
- ✅ Theta waves (4-8 Hz) - Drowsy, meditation
- ✅ Alpha waves (8-13 Hz) - Relaxed, calm
- ✅ Beta waves (13-30 Hz) - Alert, focused

### 🔬 Frequency Analysis Skills
- ✅ Convert time-domain signals to frequency-domain using FFT
- ✅ Compute power spectra to see frequency content
- ✅ Calculate band power for each frequency range
- ✅ Visualize power distribution across bands

### 🧠 EEG Interpretation
- ✅ Identify mental states from band power distribution
- ✅ Understand mixed-frequency signals
- ✅ Track sleep stages using brain waves
- ✅ Build simple mental state classifiers

### 🛠️ Practical Applications
- ✅ Sleep monitoring and analysis
- ✅ Meditation quality assessment
- ✅ Mental state classification
- ✅ Brain-computer interface basics

## 🚀 What's Next?

**Next Notebook**: `04_Signal_Cleaning.ipynb` - Learn to clean noisy biosignals using filters!

## 💡 Real-World Applications

The skills you learned are used in:
- 🏥 Sleep clinics and disorder diagnosis
- 🧘 Meditation apps and neurofeedback
- 🎮 Brain-computer interfaces (BCI)
- 🔬 Neuroscience research
- 📱 Consumer EEG headbands
- 💊 Epilepsy monitoring and treatment

---

### 📝 Key Takeaways

> **FFT (Fast Fourier Transform)**: Converts signals from time to frequency domain
>
> **Power Spectrum**: Shows the strength of each frequency in a signal
>
> **Band Power**: Total power within a specific frequency range (e.g., alpha band)
>
> **Mental States**: Different activities produce different brain wave patterns

**Keep exploring the amazing world of brain signals!** 🧠✨

---

*Made with ❤️ for curious minds by the Delta-Predictive-Biosensing team*